In [1]:
# ---
# Imports and Setup
# ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display options
pd.set_option("display.max_columns", 100)
sns.set_theme("notebook")

In [3]:
import os
os.getcwd()

'/Users/stefannhs/Projects/open-rec-sphere/notebooks'

In [5]:
# ---
# Imports and Setup
# ---

path = "../data/raw/expedia/"
df_train = pd.read_csv(path + "train.csv")
df_destinations = pd.read_csv(path + "destinations.csv")

In [23]:
# ---
# Overview
# ---

df_train.shape

(37670293, 24)

In [24]:
df_train.head()

,date_time,site_name,posa_continent,user_location_country,user_location_region,user_location_city,orig_destination_distance,user_id,is_mobile,is_package,channel,srch_ci,srch_co,srch_adults_cnt,srch_children_cnt,srch_rm_cnt,srch_destination_id,srch_destination_type_id,is_booking,cnt,hotel_continent,hotel_country,hotel_market,hotel_cluster
0,2014-08-11 07:46:59,2,3,66,348,48862,2234.2641,12,0,1,9,2014-08-27,2014-08-31,2,0,1,8250,1,0,3,2,50,628,1
1,2014-08-11 08:22:12,2,3,66,348,48862,2234.2641,12,0,1,9,2014-08-29,2014-09-02,2,0,1,8250,1,1,1,2,50,628,1
2,2014-08-11 08:24:33,2,3,66,348,48862,2234.2641,12,0,0,9,2014-08-29,2014-09-02,2,0,1,8250,1,0,1,2,50,628,1
3,2014-08-09 18:05:16,2,3,66,442,35390,913.1932,93,0,0,3,2014-11-23,2014-11-28,2,0,1,14984,1,0,1,2,50,1457,80
4,2014-08-09 18:08:18,2,3,66,442,35390,913.6259,93,0,0,3,2014-11-23,2014-11-28,2,0,1,14984,1,0,1,2,50,1457,21


In [25]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37670293 entries, 0 to 37670292
Data columns (total 24 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   date_time                  object 
 1   site_name                  int64  
 2   posa_continent             int64  
 3   user_location_country      int64  
 4   user_location_region       int64  
 5   user_location_city         int64  
 6   orig_destination_distance  float64
 7   user_id                    int64  
 8   is_mobile                  int64  
 9   is_package                 int64  
 10  channel                    int64  
 11  srch_ci                    object 
 12  srch_co                    object 
 13  srch_adults_cnt            int64  
 14  srch_children_cnt          int64  
 15  srch_rm_cnt                int64  
 16  srch_destination_id        int64  
 17  srch_destination_type_id   int64  
 18  is_booking                 int64  
 19  cnt                        int64  
 20  

## Column Classification Overview

Before diving into deeper analysis, we define and group the columns in `train.csv` based on their data types and functional roles.

This allows us to:
- Structure our EDA by column type
- Apply targeted preprocessing strategies later
- Document our assumptions about how the data behaves

We'll classify columns as:
- **Numerical**
- **Categorical**
- **Binary / Boolean**
- **Datetime**
- **ID / Structural**
- **Target features**

In [53]:
# --- Define column groups based on semantics and dtype ---

# Continuous numerical features
numerical_cols = [
    "orig_destination_distance",
    "srch_adults_cnt",
    "srch_children_cnt",
    "srch_rm_cnt",
    "cnt",
]

# Categorical features (IDs that represent groups or categories)
categorical_cols = [
    "site_name",
    "posa_continent",
    "user_location_country",
    "user_location_region",
    "user_location_city",
    "channel",
    "hotel_continent",
    "hotel_country",
    "hotel_market",
]

# Binary flags (0 or 1)
binary_cols = [
    "is_mobile",
    "is_package",
    "is_booking",
]

# Date and time features
datetime_cols = [
    "date_time",
    "srch_ci",
    "srch_co",
]

# IDs and label (used for joining, indexing, and modeling)
id_cols = [
    "user_id",
    "srch_destination_id",
    "srch_destination_type_id",
]

# Target features
target_cols = [
    "hotel_cluster", # TARGET LABEL
]

In [34]:
df_train[numerical_cols].describe()

,orig_destination_distance,srch_adults_cnt,srch_children_cnt,srch_rm_cnt,cnt
count,2.414529e+07,3.767029e+07,3.767029e+07,3.767029e+07,3.767029e+07
mean,1.970090e+03,2.024296e+00,3.321222e-01,1.112663e+00,1.483384e+00
std,2.232442e+03,9.116678e-01,7.314981e-01,4.591155e-01,1.219776e+00
min,5.600000e-03,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,3.131670e+02,2.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
50%,1.140491e+03,2.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
75%,2.552599e+03,2.000000e+00,0.000000e+00,1.000000e+00,2.000000e+00
max,1.240790e+04,9.000000e+00,9.000000e+00,8.000000e+00,2.690000e+02


In [39]:
for col in categorical_cols:
    n_unique = df_train[col].nunique()
    print(f"{col}: {n_unique} unique values")


site_name: 45 unique values
posa_continent: 5 unique values
user_location_country: 237 unique values
user_location_region: 1008 unique values
user_location_city: 50447 unique values
channel: 11 unique values
hotel_continent: 7 unique values
hotel_country: 213 unique values
hotel_market: 2118 unique values


In [40]:
for col in categorical_cols:
    print(f"\n📊 {col} — Top 5 categories:")
    print(df_train[col].value_counts().head(5))


📊 site_name — Top 5 categories:
site_name
2     23790351
11     2605866
24     2363595
37     2013818
34     1784564
Name: count, dtype: int64

📊 posa_continent — Top 5 categories:
posa_continent
3    28240462
1     4458076
2     3515919
4     1190726
0      265110
Name: count, dtype: int64

📊 user_location_country — Top 5 categories:
user_location_country
66     20346844
205     4188283
3       2212572
69      1931466
77       948722
Name: count, dtype: int64

📊 user_location_region — Top 5 categories:
user_location_region
174    4132663
348    1873377
354    1708208
442    1432740
220    1365143
Name: count, dtype: int64

📊 user_location_city — Top 5 categories:
user_location_city
5703     762261
48862    617937
25315    400451
24103    394731
36086    354610
Name: count, dtype: int64

📊 channel — Top 5 categories:
channel
9    20881299
0     4685201
1     3819309
2     2966352
5     2326077
Name: count, dtype: int64

📊 hotel_continent — Top 5 categories:
hotel_continent
2    197776

## 1. Dataset Familiarization

### Q: What does each row in train.csv represent — a search, a session, or a single interaction?

From the dataset description:

We're looking at logs of customer behavior, which include:
- what customers searched for,
- how they interacted with search results (click/book),
- whether or not the search result was a travel package.

Training data includes all the users in the logs, including both click events and booking events.

Here is the flow that generates the training data:

1. User searches for hotels (with search parameters)
2. Expedia returns a set of hotel options
3. Each hotel option is a row in the training dataset
4. A user can interact with an option (e.g. click, book) - this is tracked

### Q: What is hotel_cluster? Why are there 100 of them?

#### What is hotel_cluster?

`hotel_cluster` is a **categorical label** assigned to each hotel in the dataset, created by **Expedia’s internal clustering algorithms**. These clusters group similar hotels based on:

- Historical booking behavior
- Price levels
- Star ratings
- Geographic proximity (e.g., distance to city center)
- Possibly other latent features

This clustering allows Expedia to **reduce the complexity** of modeling individual hotels and instead focus on groups that behave similarly in terms of user preferences and booking patterns.

#### Why are there 100 of them?

We assume Expedia originally had **many more clusters**, but for the purposes of the Kaggle competition, they:

- **Randomly sampled 100 hotel clusters** from their full set
- Designed the challenge to be a **100-class classification problem**

This constraint makes the problem more tractable for participants, while still simulating a real-world multi-class recommendation scenario.



### Q: Do all columns have intuitive meanings? What about `user_location_city` vs `orig_destination_distance`?

#### General observation

While the column names are fairly descriptive, they are **internally-defined abstractions** rather than human-readable values — so **some semantics are hidden** behind ID mappings (e.g., `user_location_city`, `hotel_market`, `srch_destination_id`). You can interpret many of them through inference and exploratory analysis, but you don’t get explicit labels like "Amsterdam" or "New York".

Expedia did provide a data field description, but **interpretation still requires domain reasoning** — especially for interactions between location, time, and booking behavior.

#### Focus: `user_location_city` vs `orig_destination_distance`

|  Feature |  Description  | Data Type  |
|----------|---------------|------------|
|  `user_location_city`  |  The ID of the city the customer is located  |  int  |
|  `orig_destination_distance`  |  Physical distance between a hotel and a customer at the time of search. A null means the distance could not be calculated  |  double  |

##### Why it matters?
`user_location_city` is a **categorical variable**, and useful for understanding **home-base** or regional trends (e.g. Dutch users booking hotels in Spain during summer).

`orig_destination_distance` is a **continuous numerical variable**, and useful for identifying domestic vs international trips, travel intent, or travel planning behavior.

Their relationship can reveal latent clusters in travel behavior — like whether short-distance searches tend to be booked more often (weekend trips vs intercontinental vacations).




### Q: How many columns in destinations.csv? What's the shape and dtype?

In [22]:
df_destinations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62106 entries, 0 to 62105
Columns: 150 entries, srch_destination_id to d149
dtypes: float64(149), int64(1)
memory usage: 71.1 MB


The destinations.csv file contains:

- **62,106 rows** — each representing a unique `srch_destination_id`
- **150 columns** in total:
    - `srch_destination_id` (type: `int64`)
    - 149 feature columns named d1 to d149 (type: `float64`)
- The dataset uses **~71MB of memory** in its raw form

These 149 $d_{n}$ columns are dense numerical embeddings or latent features that likely capture characteristics of the destination.

Since they’re floats, they could represent:
- Embedding vectors from historical user behavior
- Aggregated signals (e.g., popularity, price profile, geographic features)

## 2. Data Integrity Checks
<!-- 
🧪 Hints:

Are there any missing values? If so, are they ignorable or structural?

Check for duplicates — are there repeated ids or target features?

Are all hotel clusters between 0–99? Any unexpected values?
-->


In [49]:
print(df_train["orig_destination_distance"].isnull().sum() / len(df_train.index) * 100)

35.903625703150226


In [45]:
# check missing values on complete dataset
df_train.isnull().sum()

date_time                           0
site_name                           0
posa_continent                      0
user_location_country               0
user_location_region                0
user_location_city                  0
orig_destination_distance    13525001
user_id                             0
is_mobile                           0
is_package                          0
channel                             0
srch_ci                         47083
srch_co                         47084
srch_adults_cnt                     0
srch_children_cnt                   0
srch_rm_cnt                         0
srch_destination_id                 0
srch_destination_type_id            0
is_booking                          0
cnt                                 0
hotel_continent                     0
hotel_country                       0
hotel_market                        0
hotel_cluster                       0
dtype: int64

In [50]:
df_train[df_train['is_booking'] == 1].isnull().sum()

date_time                          0
site_name                          0
posa_continent                     0
user_location_country              0
user_location_region               0
user_location_city                 0
orig_destination_distance    1015179
user_id                            0
is_mobile                          0
is_package                         0
channel                            0
srch_ci                            0
srch_co                            0
srch_adults_cnt                    0
srch_children_cnt                  0
srch_rm_cnt                        0
srch_destination_id                0
srch_destination_type_id           0
is_booking                         0
cnt                                0
hotel_continent                    0
hotel_country                      0
hotel_market                       0
hotel_cluster                      0
dtype: int64

In [51]:
df_train[df_train['is_booking'] == 0].isnull().sum()

date_time                           0
site_name                           0
posa_continent                      0
user_location_country               0
user_location_region                0
user_location_city                  0
orig_destination_distance    12509822
user_id                             0
is_mobile                           0
is_package                          0
channel                             0
srch_ci                         47083
srch_co                         47084
srch_adults_cnt                     0
srch_children_cnt                   0
srch_rm_cnt                         0
srch_destination_id                 0
srch_destination_type_id            0
is_booking                          0
cnt                                 0
hotel_continent                     0
hotel_country                       0
hotel_market                        0
hotel_cluster                       0
dtype: int64

### Q: Are there any missing values?

We analyzed the training dataset to identify missing values and determine whether they are structural (i.e., expected and explainable) or ignorable.

### Summary of Missing Values

| Column                    | Missing Values       | % Missing     | Notes |
|---------------------------|----------------------|----------------|-------|
| `orig_destination_distance` | 13,525,001             | ~35.90%        | Missing when distance couldn't be calculated |
| `srch_ci` (check-in date)   | 47,083                 | ~0.125%        | Only missing for non-bookings |
| `srch_co` (check-out date)  | 47,084                 | ~0.125%        | Only missing for non-bookings |

#### Filtering by `is_booking`

- When `is_booking == 0` (no booking):  
  - `orig_destination_distance` missing: ~33.21%  
  - `srch_ci` and `srch_co`: ~0.125% missing

- When `is_booking == 1` (booking occurred):  
  - `orig_destination_distance` missing: ~2.69%  
  - `srch_ci` and `srch_co`: ✅ no missing values

#### Interpretation

##### `orig_destination_distance`
- Likely inferred via IP geolocation or internal metadata
- Missing values are **structural**: distance couldn't be computed at the time
- High missingness suggests cautious use in modeling
- Could be:
  - Dropped entirely
  - Imputed or bucketed
  - Used with a binary flag `is_known_distance`

##### `srch_ci` and `srch_co`
- Never missing for bookings — only absent when a hotel wasn’t booked
- Their presence depends on user action
- These are **structural nulls**, not data quality issues
- Should be excluded from predictive models that run before booking happens
- There is a difference of 1, which is a one-off anomaly, **not a pattern**


#### All other columns are complete

- No missing values in categorical, binary, or label columns

#### Minor Anomaly: `srch_ci` vs `srch_co` Missing Values

While inspecting missing values in the check-in and check-out date columns, we found a curious anomaly:

| Column     | Missing Count |
|------------|----------------|
| `srch_ci`  | 47,083         |
| `srch_co`  | 47,084         |

This **1-row difference** means there's exactly one instance where a row contains a `srch_ci` (check-in date) but is missing `srch_co` (check-out date).

##### Potential Explanations:
- A logging or ingestion error
- An edge case in Expedia’s UI flow (e.g., check-in date selected but not check-out)
- An A/B variant or platform-specific behavior
- A subtle intentional anomaly from Expedia's data anonymization pipeline

This is a **non-structural, one-off issue**, not a widespread inconsistency.

##### 🧠 What We Did:
We identified this row using:

```python
df_train[df_train['srch_ci'].notnull() & df_train['srch_co'].isnull()]
```

#### Modeling Implications

##### `orig_destination_distance`

In many cases (especially when `is_booking == 0`), Expedia couldn't compute it — e.g., due to anonymized sessions, IP masking, or missing location data.


|Strategy | Why It’s Useful |
|---------|-----------------|
| **Impute with zero or mean** | Quick solution to prevent model errors, especially with tree-based models that handle skew better. |
| **Bucket the values** | Convert distance into categories like: `local`, `domestic`, `international` — this helps model general travel patterns. |
| **Add a binary flag** (`is_known_distance`) | Helps model know when a distance was available — this can carry behavioral signal. |
| **Use null-aware models** | Frameworks like LightGBM or CatBoost handle missing values natively and may not require imputation. |
| **Exclude for early models** | If you're prototyping a baseline recommender, skip this initially to reduce complexity. |

##### `srch_ci` and `srch_co` (Check-in / Check-out dates)

For the majority of search results that were not booked, Expedia never captured the check-in/check-out date.

|Strategy | Why It’s Useful |
|---------|-----------------|
| **Drop from modeling** | These values are only known after a booking is completed — not usable at inference time. |
| **Parse and extract features only from known rows** | You can engineer interesting features like `trip_duration` or `day_of_week` for session analysis or dashboarding. |
| **Use only in post-hoc analysis** | For evaluating booking behavior, seasonality, lead time, etc., but not in pre-booking models. |
| **Use for evaluation/label delay simulation** | These columns could be used in future simulation of delayed feedback loops. |

### Q: Check for duplicates in ID's and target feature Labels

Duplicate rows in the Expedia dataset are expected and present, and are often meaningful, not errors.

- Each row = one hotel listing shown for one search session
- So a user (`user_id`) appear multiple times:
    - across multiple searches
    - across multiple results within a search
- Similarly, `hotel_cluster` will repeat a lot, because it's a label associated with hotel groups, not unique hotels. To be exact, there are 100 unique hotel clusters available. This aligns with the dataset description - i.e. Expedia designed the challenge to be a 100-class classification problem.

So yes — duplicates are present in:
- `user_id` (1,198,786)
- `srch_destination_id` (59,455)
- `srch_destination_type_id` (10)
- `hotel_cluster` (100)

When we look at unique user-hotel-cluster interactions, we notice that **~40%** (14,865,110 from 37,670,293 interactions) is unique. This results in a user interacts, on average, with **2.53** hotel clusters.

In [90]:
# Duplicate users? True = yes, False = no
df_train["user_id"].duplicated().any()

user_id
1187360    530
1040395    501
124565     498
1043120    495
783124     491
Name: count, dtype: int64

In [91]:
# Duplicate hotel clusters? True = yes, False = no
df_train["hotel_cluster"].duplicated().any()

np.True_

In [101]:
# Distinct values, non-null count, count all values in hotel clusters

for col in id_cols + target_cols:
    print(f"Column: {col}")
    print(df_train[col].agg(['nunique','count','size']))
    print("----------")

Column: user_id
nunique     1198786
count      37670293
size       37670293
Name: user_id, dtype: int64
----------
Column: srch_destination_id
nunique       59455
count      37670293
size       37670293
Name: srch_destination_id, dtype: int64
----------
Column: srch_destination_type_id
nunique          10
count      37670293
size       37670293
Name: srch_destination_type_id, dtype: int64
----------
Column: hotel_cluster
nunique         100
count      37670293
size       37670293
Name: hotel_cluster, dtype: int64
----------


In [126]:
# Count how many times each user interacted with each hotel_cluster
n_user_hotel_cluster_interactions = df_train.groupby(["user_id", "hotel_cluster"]).size()

# Total interactions across all user-cluster pairs
total_user_hotel_cluster_interactions = n_user_hotel_cluster_interactions.sum()

# Number of unique user-cluster pairs
unique_user_hotel_cluster_interactions = n_user_hotel_cluster_interactions.size

# Print summary
print(f"Total (repeated) user-cluster interactions: {total_user_hotel_cluster_interactions:,}")
print(f"Unique user-cluster pairs: {unique_user_hotel_cluster_interactions:,}")
print(f"Percentage unique user-cluster interactions: {(unique_user_hotel_cluster_interactions / total_user_hotel_cluster_interactions) * 100:.2f}%")
print(f"Average interactions per user-cluster pair: {total_user_hotel_cluster_interactions / unique_user_hotel_cluster_interactions:.2f}")

Total (repeated) user-cluster interactions: 37,670,293
Unique user-cluster pairs: 14,865,110
Percentage unique user-cluster interactions: 39.46
Average interactions per user-cluster pair: 2.53


### Q: Are all hotel clusters between 0–99? Any unexpected values?

✅ Yes — all values in the `hotel_cluster` column fall within the expected range of **0 to 99**.

- `hotel_cluster` is a numerical class label assigned by Expedia to group similar hotels.
- There are exactly **100 unique clusters**, as expected.
- No missing values or out-of-bound entries were found.

This confirms:
- The label space is **bounded and clean**
- It’s safe to treat this column as a **target for classification or ranking**
- No additional mapping or filtering is needed


In [146]:
print(f"Number of unique hotel clusters: {df_train['hotel_cluster'].nunique()}") # should return 100
print(f"Hotel cluster range: {df_train['hotel_cluster'].min()} - {df_train['hotel_cluster'].max()}") # should return 0-99

Number of unique hotel clusters: 100
Hotel cluster range: 0 - 99


## 3. Date and Time Exploration
<!--
⏱️ Hints:

What’s the range of date_time?

Can you extract month, day, day-of-week, or hour?

Do clicks/bookings cluster around certain times of year? Could seasonality matter?

Is srch_ci always before srch_co?
-->

## 4. Session and Search Behavior
<!--
💡 Hints:

Is srch_id a proxy for a session or just one result row?

How many results per srch_id? Uniform or varied?

Can you group by srch_id and see what types of hotel clusters are shown? Booked?
-->

## 5. Feature Relationships
<!-- 
🔗 Hints:

Explore relationship between user_location_country and srch_destination_id — do people often book within-country?

Which features are most correlated with bookings?

Are clicks mostly “first few rows”? Or does cluster ID matter?
-->


## 6. Class Imbalance + Target Leakage
<!--
🎯 Hints:

Do some hotel_cluster values dominate? Check their frequency.

Are booking_bool and click_bool mutually exclusive or overlapping?
What’s the distribution of click_bool, booking_bool? Class imbalance?
Is there any feature that directly reveals the target (hotel_cluster)? (Hint: look for ID columns or hashes)
-->

## 7. Embedding / Vector Analysis (destinations.csv)
<!--
🧬 Hints:

How many srch_destination_ids are there? Are some used way more than others?

Are the vectors normalized? What’s their mean/std?

Try visualizing 2D projections (later) — what shapes emerge?
-->


## 8. Distribution of Numeric Fields
<!--
📊 Hints:

Look at distributions of price_usd, orig_destination_distance, srch_length_of_stay

Are these skewed? Bimodal?

Try log-scaling and compare histograms
-->

## 9. User/Item Frequency
<!--
🧮 Hints:

Which user_ids appear the most?

Are user_ids unique to srch_ids or reused?

Same with hotel_cluster — are certain clusters disproportionately represented?

What’s the distribution of click_bool, booking_bool? Class imbalance?
-->


## 10. Building Intuition for Features
<!--
💭 Hints:

How might srch_saturday_night_bool influence booking likelihood?

How could user_location_region and srch_destination_id relate?

What other combinations (e.g., hotel_country + hotel_market) might influence choice?
-->